# Testing frameworks: `unittest`

[Documentation](https://docs.python.org/3/library/unittest.html)

`unittest` is a Python Standard Librabry module which provides a rich set of tools for constructing and running tests.

To achieve this, `unittest` supports some important concepts in an object-oriented way:

* **test fixture**
A test fixture represents the preparation needed to perform one or more tests, and any associate cleanup actions. This may involve, for example, creating temporary or proxy databases, directories, or starting a server process.

* **test case**
A test case is the individual unit of testing. It checks for a specific response to a particular set of inputs. unittest provides a base class, TestCase, which may be used to create new test cases.

* **test suite**
A test suite is a collection of test cases, test suites, or both. It is used to aggregate tests that should be executed together.

* **test runner**
A test runner is a component which orchestrates the execution of tests and provides the outcome to the user. The runner may use a graphical interface, a textual interface, or return a special value to indicate the results of executing the tests.


The building block of a test is the test case. With `unittest`, you can create a test case by subclassing the `unittest.TestCase` class. Each method starting with `test` will be an individual testing scenario. Test methods should contain at least one `assert*` method call as this is the essential role of a test: comparing actual results agaist expected results.

In [2]:
import unittest


class TestStringMethods(unittest.TestCase):

    def test_upper(self):
        self.assertEqual('foo'.upper(), 'FOO')

    def test_isupper(self):
        self.assertTrue('FOO'.isupper())
        self.assertFalse('Foo'.isupper())

    def test_split(self):
        s = 'hello world'
        self.assertEqual(s.split(), ['hello', 'world'])
        # check that s.split fails when the separator is not a string
        with self.assertRaises(TypeError):
            s.split(2)


unittest.main(argv=['-k', 'TestStringMethods'], verbosity=2, exit=False)

test_isupper (__main__.TestStringMethods) ... ok
test_split (__main__.TestStringMethods) ... ok
test_upper (__main__.TestStringMethods) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.004s

OK


The following table lists the most commonly used assert methods:


| Method                                  | Checks that                       |
| --------------------------------------- | --------------------------------- |
| `assertEqual(a, b)`                     | `a == b`                          |
| `assertNotEqual(a, b)`                  | `a != b`                          |
| `assertTrue(x)`                         | `bool(x) is True`                 |
| `assertFalse(x)`                        | `bool(x) is False`                |
| `assertIs(a, b)`                        | `a is b`                          |
| `assertIsNot(a, b)`                     | `a is not b`                      |
| `assertIsNone(x)`                       | `x is None`                       |
| `assertIsNotNone(x)`                    | `x is not None`                   |
| `assertIn(a, b)`                        | `a in b`                          |
| `assertNotIn(a, b)`                     | `a not in b`                      |
| `assertIsInstance(a, b)`                | `isinstance(a, b)`                |
| `assertNotIsInstance(a, b)`             | `not isinstance(a, b)`            |
| `assertRaises(exc, fun, *args, **kwds)` | `fun(*args, **kwds)` raises `exc` |


Test fixtures represent the initial set-up needed before each test method or
before all the tests in a test case. This can be achieved by using the special
`setUp` method that will be called before every test run. Similarly, we can 
provide a `tearDown()` method that tidies up after the test method has been
run.


```python
import unittest

class WidgetTestCase(unittest.TestCase):
    def setUp(self):
        self.widget = Widget('The widget')

    def test_default_widget_size(self):
        self.assertEqual(self.widget.size(), (50,50),
                         'incorrect default size')

    def test_widget_resize(self):
        self.widget.resize(100,150)
        self.assertEqual(self.widget.size(), (100,150),
                         'wrong size after resize')

    def tearDown(self):
        self.widget.dispose()
```

### Command-Line Interface

The unittest module can be used from the command line to run tests from modules, classes or even individual test methods:

```shell
python -m unittest test_module1 test_module2
python -m unittest test_module.TestClass
python -m unittest test_module.TestClass.test_method
```

If you with to stop test run on the first error or failure, run the tests with
`-f, --failfast` command-line option:

```shell
python -m unittest -f
python -m unittest --failfast
```

For a list of all the command-line options:

```shell
python -m unittest -h
```

### Test Suites

It is recommended that you use `TestCase` implementations to group tests together according to the features they test. `unittest` provides another mechanism for grouping tests: **the test suite**, represented by `unittest`’s `TestSuite` class. In most cases, calling unittest.main() will do the right thing and collect all the module’s test cases for you and execute them.

However, should you want to customize the building of your test suite, you can do it yourself:

```python
def suite():
    suite = unittest.TestSuite()
    suite.addTest(WidgetTestCase('test_default_widget_size'))
    suite.addTest(WidgetTestCase('test_widget_resize'))
    return suite

if __name__ == '__main__':
    runner = unittest.TextTestRunner()
    runner.run(suite())
```

### Subtests

When there are very small differences among your tests, for instance some parameters, unittest allows you to distinguish them inside the body of a test method using the `subTest()` context manager.


In [2]:
import unittest


class NumbersTest(unittest.TestCase):

    def test_even(self):
        """
        Test that numbers between 0 and 5 are all even.
        """
        for i in range(0, 6):
            with self.subTest(f"Test failed for i = {i}"):
                self.assertEqual(i % 2, 0)
 

unittest.main(argv=['-k', 'NumbersTest'], verbosity=2, exit=False)

test_even (__main__.NumbersTest)
Test that numbers between 0 and 5 are all even. ... 
FAIL: test_even (__main__.NumbersTest) [Test failed for i = 1]
Test that numbers between 0 and 5 are all even.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/var/folders/s2/_752pzb50tg00mv9ggtr6p8c0000gn/T/ipykernel_66285/1847049937.py", line 12, in test_even
    self.assertEqual(i % 2, 0)
AssertionError: 1 != 0

FAIL: test_even (__main__.NumbersTest) [Test failed for i = 3]
Test that numbers between 0 and 5 are all even.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/var/folders/s2/_752pzb50tg00mv9ggtr6p8c0000gn/T/ipykernel_66285/1847049937.py", line 12, in test_even
    self.assertEqual(i % 2, 0)
AssertionError: 1 != 0

FAIL: test_even (__main__.NumbersTest) [Test failed for i = 5]
Test that numbers between 0 and 5 are all even.
-------------------------------------

### Skipping tests and expected failures

Unittest supports skipping individual test methods and even whole classes of tests. In addition, it supports marking a test as an “expected failure,” a test that is broken and will fail, but shouldn’t be counted as a failure on a TestResult.

In [3]:
import sys
import unittest

VERSION = (1, 2)

def external_resource_available():
    return False


class MyTestCase(unittest.TestCase):

    @unittest.skip("demonstrating skipping")
    def test_nothing(self):
        self.fail("shouldn't happen")

    @unittest.skipIf(VERSION < (1, 3),
                     "not supported in this library version")
    def test_format(self):
        # Tests that work for only a certain version of the library.
        pass

    @unittest.skipUnless(sys.platform.startswith("win"), "requires Windows")
    def test_windows_support(self):
        # windows specific testing code
        pass

    def test_maybe_skipped(self):
        if not external_resource_available():
            self.skipTest("external resource not available")
        # test code that depends on the external resource
        pass
    
    @unittest.expectedFailure
    def test_fail(self):
        self.assertEqual(1, 0, "broken")
    
    
unittest.main(argv=['-k', 'MyTestCase'], verbosity=2, exit=False)

test_fail (__main__.MyTestCase) ... expected failure
test_format (__main__.MyTestCase) ... skipped 'not supported in this library version'
test_maybe_skipped (__main__.MyTestCase) ... skipped 'external resource not available'
test_nothing (__main__.MyTestCase) ... skipped 'demonstrating skipping'
test_windows_support (__main__.MyTestCase) ... skipped 'requires Windows'

----------------------------------------------------------------------
Ran 5 tests in 0.004s

OK (skipped=4, expected failures=1)
